# Проект: рынок видеоигр

### Цели и задачи проекта

Цель проекта — познакомиться с данными, проверить их корректность и провести предобработку, получив необходимый срез данных.

Перед анализом дополнительно сделаем следующее:

* Отберем данные по времени выхода игры. Нам нужен период с 2000 по 2013 год включительно.
* Категоризуем игры по оценкам пользователей и экспертов. Выделим три категории:
    * высокая оценка — с оценкой от 8 до 10 и от 80 до 100, включая правые границы интервалов.
    * средняя оценка — с оценкой от 3 до 8 и от 30 до 80, не включая правые границы интервалов.
    * низкая оценка — с оценкой от 0 до 3 и от 0 до 30, не включая правые границы интервалов.
* Выделим топ-7 платформ по количеству игр, выпущенных за весь требуемый период.


## 1. Загрузка данных и знакомство с ними


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('https://code.s3.yandex.net//datasets/new_games.csv')

In [3]:
df_first = pd.read_csv('https://code.s3.yandex.net//datasets/new_games.csv')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Нам предоставлены данные, в которых: 11 столбцов, 16956 записей, диапазон индексов: от 0 до 16955. Названия столбцов соответствуют описанию, которое мы получили. 

**Пропуски встречаются в столбцах:** 
* Name
* Year of Release
* Genre
* Critic Score 
* User Score 
* Rating

Наибольшие проблемы с рейтингами и оценками:
- Critic Score — пропущено более половины значений (51,4 %).
- User Score и Rating — пропущено около 40 %.

Минимальные пропуски в Name и Genre (по 2 записи).

Полные данные (без пропусков) в столбцах: Platform, NA sales, EU sales, JP sales, Other sales.

**Использованы некорректные типы данных в столбцах:** 
* Year of Release (float64, а должен быть int64 или datetime64)
* EU sales (object, а должен быть float64)
* JP sales (object, а должен быть float64)
* User Score (object, а должен быть float64)

Genre и Rating можно перевести в тип category для экономии памяти.

**Дополнительные особенности данных, которые важно учесть при предобработке:**

* Name — корректно, но можно уточнить как Game_Name для однозначности;

* Year of Release — содержит пробел и предлог, что неудобно для работы в коде;

* NA sales, EU sales, JP sales, Other sales — содержат пробелы, аббревиатура без пояснения;

* Rating — общее название, неясно, какой рейтинг (возрастной, пользовательский и т. д.).

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма

Выводим на экран названия всех столбцов датафрейма и проверяем их стиль написания:

In [5]:
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')


Приводим все столбцы к стилю snake case. Названия должны быть в нижнем регистре, а вместо пробелов — подчёркивания.

In [6]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')


### 2.2. Типы данных

**Возможные причины некорректного типа данных:**
* EU sales и JP sales — тип object вместо float64
    * текстовые обозначения вместо чисел («N/A», «unknown», «—», «null»)

    * разделители тысяч в виде запятых (1,000,000 вместо 1000000)

    * пробелы или символы валют в строках ($10M, 10 млн)

    * смешанный формат: часть значений — числа, часть — строки

    * использование запятой как десятичного разделителя (10,5 вместо 10.5)
    
* User Score — тип object вместо float64
    * значения «tbd» (to be determined) или «N/A» вместо пропусков;

    * запятая вместо точки в десятичных числах (8,5 → должно быть 8.5);

    * текстовые оценки («high», «medium», «low»);

    * дополнительные пояснения в ячейках (8.5 (based on 10 reviews)).
* Year of Release — тип float64 вместо int64 или datetime64
    * наличие пропусков (NaN) в столбце. В pandas значения NaN представлены как float, поэтому весь столбец автоматически приводится к float64, даже если остальные значения — целые числа.


Меняем тип данных в столбцах:
* year_of_release
* eu_sales
* jp_sales
* user_score

In [7]:
# заменяем NaN в 'year_of_release' на ноль, меняем тип данных с float на integer, понижаем разрядность
df['year_of_release'] = df['year_of_release'].fillna(0)
df['year_of_release'] = pd.to_numeric(df['year_of_release'], downcast='integer')

# меняем тип данных в остальных столбцах, понижаем разрядность
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce', downcast='float')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce', downcast='float')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce', downcast='float')


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16956 non-null  int16  
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float32
 6   jp_sales         16952 non-null  float32
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float32
 10  rating           10085 non-null  object 
dtypes: float32(3), float64(3), int16(1), object(4)
memory usage: 1.1+ MB


Все нужные столбы, которые были в типе object стали int/float, а это значит, что теперь с ними можно будет производить специальные операции, характерные для числовых столбцов.

### 2.3. Наличие пропусков в данных

In [9]:
# Подсчитываем количество пропусков в каждом столбце
print(df.isna().sum().sort_values(ascending=False))

user_score         9268
critic_score       8714
rating             6871
eu_sales              6
jp_sales              4
name                  2
genre                 2
platform              0
year_of_release       0
na_sales              0
other_sales           0
dtype: int64


In [10]:
# Подсчитываем долю строк с пропусками
share = df.isna().sum() / len(df) * 100
print(share.sort_values(ascending=False))

user_score         54.659118
critic_score       51.391838
rating             40.522529
eu_sales            0.035386
jp_sales            0.023590
name                0.011795
genre               0.011795
platform            0.000000
year_of_release     0.000000
na_sales            0.000000
other_sales         0.000000
dtype: float64


**Промежуточный вывод по пропускам в данных**

Столбцы с наибольшим числом пропусков
* user_score — 54,66 % пропусков.
* critic_score — 51,39 % пропусков.
* rating — 40,52 % пропусков.
* year_of_release — 1,62 % пропусков.

**Столбцы с единичными пропусками или без них**
* eu_sales — 0,035 % (6 пропусков);

* jp_sales — 0,024 % (4 пропуска);

* name — 0,012 % (2 пропуска);

* genre — 0,012 % (2 пропуска);

* platform, na_sales, other_sales — пропусков нет (0 %).

**Почему могли возникнуть пропуски**
1. user_score и critic_score (более 50 % пропусков):

* отсутствие оценок для старых игр (до появления агрегаторов);

* малоизвестность игры — не набрала достаточного числа оценок;

* игра вышла недавно — ещё не получила оценок;

* данные не были собраны для некоторых платформ или регионов;

* технические ошибки при парсинге данных с сайтов‑агрегаторов.

2. rating (40,5 %):

* разные системы рейтингов в разных странах (ESRB, PEGI и др.) — не всегда есть соответствие;

* для старых игр рейтинги могли не присваиваться;

* региональные различия: в некоторых странах рейтинг отсутствует;

* отсутствие рейтинга для инди‑игр или малоизвестных проектов.

3. year_of_release (1,62 %):

* ошибки ввода данных или опечатки;

* игры с неопределённой датой выхода (например, в раннем доступе);

* старые игры, для которых точная дата неизвестна;

* сбор данных из источников с неполной информацией.

4. Единичные пропуски (eu_sales, jp_sales, name, genre):

* опечатки или ошибки ввода при заполнении данных;

* неполнота информации для отдельных игр;

* проблемы с парсингом данных для конкретных записей.

5. Столбцы без пропусков (platform, na_sales, other_sales):

* обязательные поля при сборе данных;

* высокая важность для аналитики (продажи по регионам);

* стандартизированные источники данных.

**Возможные действия с данными и обоснование**

**Для столбцов с высокой долей пропусков (> 40 %)**
1. user_score, critic_score:

* Оставить как есть — если анализ не требует этих оценок.

* Создать бинарный флаг has_score (1 — есть оценка, 0 — нет). Позволяет учитывать наличие/отсутствие оценок в моделях.

* Импутация медианой/средним — если нужно сохранить числовой формат. Подходит для простых моделей, но снижает вариативность данных.

* Группировка по категориям — например, разделить на «с оценками» и «без оценок» для сравнительного анализа.

* Исключить из анализа — если пропуски критичны и нет способа их восполнить.

2. rating:

* Заменить пропуски категорией «Unknown» — сохраняет структуру данных и позволяет анализировать влияние рейтинга.

* Объединить редкие категории — сгруппировать малочисленные рейтинги в «Other».

* Использовать как признак — создать флаг has_rating (1/0).

**Для столбца с умеренной долей пропусков (~1,6 %)**

year_of_release:

* Импутация медианным/модальным годом по жанру или платформе. Логично, если пропуски случайны.

* Предсказать год на основе других данных (жанр, платформа, название) — если есть достаточно информации.

* Удалить строки — допустимо при малом числе пропусков (276 строк из 16 956).

**Для столбцов с единичными пропусками (< 0,04 %)**
* eu_sales, jp_sales: заполнить средним/медианным значением по жанру/платформе или нулями (если продажи действительно могли быть нулевыми).

* name, genre:

* удалить строки с пропусками — потери данных минимальны (2 строки);

* попытаться восстановить из других источников (например, по ID игры).

**Общие рекомендации**
* Анализ причин пропусков — проверить, не связаны ли пропуски с определёнными группами данных (например, старые игры чаще без оценок).

* Визуализация — построить графики распределения пропусков по годам, жанрам, платформам.

* Сохранение информации о пропусках — создать дополнительные признаки (например, missing_user_score), чтобы учесть их влияние в моделях.

* Проверка на дубликаты и аномалии — перед обработкой пропусков убедиться, что данные корректны.

**Краткий итог**
* Критические пропуски (> 40 %): user_score, critic_score, rating. Требуют взвешенного подхода — импутация или исключение.

* Умеренные пропуски (~1,6 %): year_of_release. Допустима импутация или удаление.

* Единичные пропуски (< 0,04 %): eu_sales, jp_sales, name, genre. Можно удалить или заполнить простыми методами.

* Полные столбцы: platform, na_sales, other_sales. Не требуют обработки.

**Выбор оптимальных вариантов обработки для пропущенных значений**
1. user_score (54,66 % пропусков)

Заменим на индикатор -1. Значение -1 не входит в шкалу оценок (0–10), поэтому не будет перепутано с реальными данными. В дальнейшем при категоризации -1 также можно отнести к unknown.

2. critic_score (51,39 % пропусков)

Заменим на индикатор -1. Как и для user_score, -1 не встречается в шкале 0–100, что делает его безопасным индикатором.

3. rating (40,52 % пропусков)

Заменим на категорию Unknown. Сохраняет информацию о пропуске, не удаляет строки. Позволяет анализировать влияние наличия рейтинга на продажи.

4. year_of_release (1,62 % пропусков)

Уже заменили на индикатор 0 выше в коде.

5. eu_sales, jp_sales (< 0,04 % пропусков)

Заполним медианой по (platform, year_of_release). Учитывает различия в популярности платформ и динамике рынка. Для малых пропусков практически не искажает данные.

6. name, genre (< 0,02 % пропусков)

Удалим строки с пропусками. Потеря всего 2 строк из 16 956 несущественна, а удаление гарантирует корректность данных.



In [11]:
# В столбце 'user_score' пропуски заменим индикатором -1
df['user_score'] = df['user_score'].fillna(-1)

In [12]:
# В столбце 'critic_score' пропуски заменим индикатором -1

df['critic_score'] = df['critic_score'].fillna(-1)

In [13]:
# В столбце 'rating' пропуски заменим индикатором категории 'Unknown'

df['rating'] = df['rating'].fillna('Unknown')

In [14]:
# Удаляем строки с пропусками с столбцах 'name', 'genre'

df = df.dropna(subset=['name', 'genre'])

In [15]:
# заменяем пропуски в столбце 'eu_sales' средним значением на основе платформы и года выпуска

def mean_group_eu_sales(row):
    if pd.isna(row['eu_sales']):
        group = df[(df['platform'] == row['platform']) & 
               (df['year_of_release'] == row['year_of_release'])]
        return group['eu_sales'].mean()
    else:
        return row['eu_sales']
df['eu_sales'] = df.apply(mean_group_eu_sales, axis=1)

In [16]:
# заменяем пропуски в столбце 'jp_sales' средним значением на основе платформы и года выпуска

def mean_group_jp_sales(row):
    if pd.isna(row['jp_sales']):
        group = df[(df['platform'] == row['platform']) & 
               (df['year_of_release'] == row['year_of_release'])]
        return group['jp_sales'].mean()
    else:
        return row['jp_sales']
df['jp_sales'] = df.apply(mean_group_jp_sales, axis=1)

In [17]:
# проверяем пропуски по всем столбцам: пропуски отсутствуют

print(df.isna().sum().sort_values(ascending=False))

name               0
platform           0
year_of_release    0
genre              0
na_sales           0
eu_sales           0
jp_sales           0
other_sales        0
critic_score       0
user_score         0
rating             0
dtype: int64


In [18]:
# число строк с пропусками: 0

total_rows = len(df)

# Число строк с пропусками (хотя бы в одном столбце)
rows_with_na = df.isna().any(axis=1).sum()

# Процент строк с пропусками
percentage_na = (rows_with_na / total_rows) * 100

print(f"Всего строк: {total_rows}")
print(f"Строк с пропусками: {rows_with_na}")
print(f"Процент строк с пропусками: {percentage_na:.2f}%")

Всего строк: 16954
Строк с пропусками: 0
Процент строк с пропусками: 0.00%


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16954 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16954 non-null  object 
 2   year_of_release  16954 non-null  int16  
 3   genre            16954 non-null  object 
 4   na_sales         16954 non-null  float64
 5   eu_sales         16954 non-null  float64
 6   jp_sales         16954 non-null  float64
 7   other_sales      16954 non-null  float64
 8   critic_score     16954 non-null  float64
 9   user_score       16954 non-null  float32
 10  rating           16954 non-null  object 
dtypes: float32(1), float64(5), int16(1), object(4)
memory usage: 1.4+ MB


### 2.4. Явные и неявные дубликаты в данных

Уникальные значения в категориальных данных: названия жанра игры, платформы, рейтинга и года выпуска.

In [20]:
# Применяем метод unique() к столбцу 'name'
unique_name = df['name'].unique()
# Выводим результат
print(f'Уникальные значения названий игр: {unique_name}')
print()

# Применяем метод unique() к столбцу 'genre'
unique_genre = df['genre'].unique()
# Выводим результат
print(f'Уникальные значения жанров игр: {unique_genre}')
print()

# Применяем метод unique() к столбцу 'platform'
unique_platform = df['platform'].unique()
# Выводим результат
print(f'Уникальные значения платформ: {unique_platform}')
print()

# Применяем метод unique() к столбцу 'rating'
unique_rating = df['rating'].unique()
# Выводим результат
print(f'Уникальные значения рейтинга: {unique_rating}')
print()

# Применяем метод unique() к столбцу 'year_of_release'
unique_year_of_release = df['year_of_release'].unique()
# Выводим результат
print(f'Уникальные значения года выпуска: {unique_year_of_release}')

Уникальные значения названий игр: ['Wii Sports' 'Super Mario Bros.' 'Mario Kart Wii' ...
 'Woody Woodpecker in Crazy Castle 5' 'LMA Manager 2007'
 'Haitaka no Psychedelica']

Уникальные значения жанров игр: ['Sports' 'Platform' 'Racing' 'Role-Playing' 'Puzzle' 'Misc' 'Shooter'
 'Simulation' 'Action' 'Fighting' 'Adventure' 'Strategy' 'MISC'
 'ROLE-PLAYING' 'RACING' 'ACTION' 'SHOOTER' 'FIGHTING' 'SPORTS' 'PLATFORM'
 'ADVENTURE' 'SIMULATION' 'PUZZLE' 'STRATEGY']

Уникальные значения платформ: ['Wii' 'NES' 'GB' 'DS' 'X360' 'PS3' 'PS2' 'SNES' 'GBA' 'PS4' '3DS' 'N64'
 'PS' 'XB' 'PC' '2600' 'PSP' 'XOne' 'WiiU' 'GC' 'GEN' 'DC' 'PSV' 'SAT'
 'SCD' 'WS' 'NG' 'TG16' '3DO' 'GG' 'PCFX']

Уникальные значения рейтинга: ['E' 'Unknown' 'M' 'T' 'E10+' 'K-A' 'AO' 'EC' 'RP']

Уникальные значения года выпуска: [2006 1985 2008 2009 1996 1989 1984 2005 1999 2007 2010 2013 2004 1990
 1988 2002 2001 2011 1998 2015 2012 2014 1992 1997 1993 1994 1982 2016
 2003 1986 2000    0 1995 1991 1981 1987 1980 1983]


In [21]:
for column in ['platform','year_of_release','genre','rating']:
        print(f'Уникальные значения в столбце {column}:')
        print(df[column].sort_values().unique())
        print()

Уникальные значения в столбце platform:
['2600' '3DO' '3DS' 'DC' 'DS' 'GB' 'GBA' 'GC' 'GEN' 'GG' 'N64' 'NES' 'NG'
 'PC' 'PCFX' 'PS' 'PS2' 'PS3' 'PS4' 'PSP' 'PSV' 'SAT' 'SCD' 'SNES' 'TG16'
 'WS' 'Wii' 'WiiU' 'X360' 'XB' 'XOne']

Уникальные значения в столбце year_of_release:
[   0 1980 1981 1982 1983 1984 1985 1986 1987 1988 1989 1990 1991 1992
 1993 1994 1995 1996 1997 1998 1999 2000 2001 2002 2003 2004 2005 2006
 2007 2008 2009 2010 2011 2012 2013 2014 2015 2016]

Уникальные значения в столбце genre:
['ACTION' 'ADVENTURE' 'Action' 'Adventure' 'FIGHTING' 'Fighting' 'MISC'
 'Misc' 'PLATFORM' 'PUZZLE' 'Platform' 'Puzzle' 'RACING' 'ROLE-PLAYING'
 'Racing' 'Role-Playing' 'SHOOTER' 'SIMULATION' 'SPORTS' 'STRATEGY'
 'Shooter' 'Simulation' 'Sports' 'Strategy']

Уникальные значения в столбце rating:
['AO' 'E' 'E10+' 'EC' 'K-A' 'M' 'RP' 'T' 'Unknown']



Проблема: дублирование из‑за разного регистра букв в столбце 'genre'.
Решение: приводим все значения к нижнему регистру.

In [22]:
df['genre'] = df['genre'].str.lower()

# результат нормализации:
unique_genre = df['genre'].unique()
# Выводим результат
print(f'Результат нормализации, уникальные значения жанров игр: {unique_genre}')

Результат нормализации, уникальные значения жанров игр: ['sports' 'platform' 'racing' 'role-playing' 'puzzle' 'misc' 'shooter'
 'simulation' 'action' 'fighting' 'adventure' 'strategy']


В остальных столбцах проблем не обнаружено. 



In [23]:
# проверяем количество явных строк дубликатов

duplicated_rows = df.duplicated().sum() 
print(f'Количество явных строк дубликатов: {duplicated_rows}')

Количество явных строк дубликатов: 241


In [24]:
# Сортируем датафрейм по всем столбцам
df_sorted = df.sort_values(by=df.columns.tolist())

# Находим дубликаты
duplicates = df_sorted[df_sorted.duplicated(keep=False)]

# Выводим дубликаты
print(duplicates)

                                   name platform  year_of_release  \
15191                    Beyblade Burst      3DS             2016   
15192                    Beyblade Burst      3DS             2016   
15301                 11eyes: CrossOver     X360             2009   
15302                 11eyes: CrossOver     X360             2009   
4860   18 Wheeler: American Pro Trucker      PS2             2001   
...                                 ...      ...              ...   
2909   Yu-Gi-Oh! The Falsebound Kingdom       GC             2002   
6695                      Zoo Resort 3D      3DS             2011   
6696                      Zoo Resort 3D      3DS             2011   
8156                 Zumba Fitness Rush     X360             2012   
8157                 Zumba Fitness Rush     X360             2012   

              genre  na_sales  eu_sales  jp_sales  other_sales  critic_score  \
15191  role-playing      0.00      0.00      0.03         0.00          -1.0   
15192  role

In [25]:
# теперь воспользуемся методом drop_duplicates() для удаления дубликатов:

# сохраняем количество строк до удаления строк (df_first - самый первый df до всех манипуляций с удалениями)
initial_row_count = df_first.shape[0]

# удаляем дубликаты из текущего датафрейма df и записываем его в новую переменную df_no_duplicates
df_no_duplicates = df.drop_duplicates()

# сохраняем количество строк после удаления дубликатов
final_row_count = df_no_duplicates.shape[0]

percentage_removed = (initial_row_count - final_row_count) / initial_row_count * 100

# выводим результаты
print(f"Количество строк до удаления дубликатов: {initial_row_count}")
print(f"Количество строк после удаления дубликатов: {final_row_count}")
print(f"Количество удаленных строк: {initial_row_count - final_row_count}")
print(f"Доля удалённых строк от общего количества, выраженная в процентах: {percentage_removed:.2f}%")



Количество строк до удаления дубликатов: 16956
Количество строк после удаления дубликатов: 16713
Количество удаленных строк: 243
Доля удалённых строк от общего количества, выраженная в процентах: 1.43%


### Общий промежуточный вывод

В ходе работы был проведён комплексный анализ и очистка датафрейма с целью подготовки данных для дальнейшего анализа. Исходный датафрейм содержал 16956 строк. Первичный осмотр выявил следующие проблемы: наличие пропусков в ряде столбцов, несоответствие типов данных ожидаемым, присутствие дубликатов (явных и неявных).

Анализ пропусков показал, что наибольшее количество пропущенных значений наблюдалось в столбцах: 
* Critic Score — пропущено более половины значений (51,4 %).
* User Score и Rating — пропущено около 40 %.

Для заполнения пропусков были применены следующие стратегии:
* Замена на индикаторы
* Удаление
* Замена медианой

Некорректные типы данных для столбцов привели к нужным:
* Year of Release (float64 -> int64)
* EU sales (object -> float64)
* JP sales (object -> float64)
* User Score (object -> float64)

Привели все столбцы к стилю snake case. Названия в нижнем регистре, а вместо пробелов — подчёркивания.

В процессе очистки были выявлены и удалены явные дубликаты и неявные дубликаты в количестве 243 строк/1.43% от общих данных.
Удаление дубликатов позволило устранить избыточность данных и повысить точность последующего анализа.


После выполнения всех этапов очистки:

* пропуски в датафрейме отсутствуют (проверено методом `isna().sum())`;

* дубликаты удалены (проверено методом `duplicated().sum())`;

* все столбцы имеют корректные типы данных и понижены до оптимальных (проверено `dtypes`).


In [26]:
# пропуски в датафрейме отсутствуют (проверено методом isna().sum());

print(df_no_duplicates.isna().sum().sort_values(ascending=False))

name               0
platform           0
year_of_release    0
genre              0
na_sales           0
eu_sales           0
jp_sales           0
other_sales        0
critic_score       0
user_score         0
rating             0
dtype: int64


In [27]:
# дубликаты удалены (проверено методом duplicated().sum());

print(df_no_duplicates.duplicated().sum())

0


In [28]:
# все столбцы имеют корректные типы данных и понижены до оптимальных (проверено dtypes)

print(df_no_duplicates.dtypes)

name                object
platform            object
year_of_release      int16
genre               object
na_sales           float64
eu_sales           float64
jp_sales           float64
other_sales        float64
critic_score       float64
user_score         float32
rating              object
dtype: object


**Данные готовы к анализу**

---

## 3. Фильтрация данных

Нас интересует период с 2000 по 2013 год включительно. Отберем данные по этому показателю. Сохраните новый срез данных в отдельном датафрейме, например `df_actual`.*

In [29]:
# отбираем данные за период с 2000 по 2013 год включительно

df_actual = df_no_duplicates[df_no_duplicates['year_of_release'].between(2000, 2013)].copy()

---

## 4. Категоризация данных
    
*Проведем категоризацию данных:*
- *Разделим все игры по оценкам пользователей и выделим такие категории: высокая оценка (от 8 до 10 включительно), средняя оценка (от 3 до 8, не включая правую границу интервала) и низкая оценка (от 0 до 3, не включая правую границу интервала).*

In [30]:
# разделяем все игры по оценкам пользователей и выделяем категории:

def categorize_user_score(grade):
    # низкая оценка (от 0 до 3, не включая правую границу интервала)
    if grade < 3:
        return 'Низкая оценка'  
    # средняя оценка (от 3 до 8, не включая правую границу интервала)
    elif grade < 8:
        return 'Средняя оценка'
    # высокая оценка (от 8 до 10 включительно)
    elif grade <= 10:
        return 'Высокая оценка'
    else:
        return 'Не определено'

df_actual['user_score_category'] = df_actual['user_score'].apply(categorize_user_score)
print(df_actual)

                                                   name platform  \
0                                            Wii Sports      Wii   
2                                        Mario Kart Wii      Wii   
3                                     Wii Sports Resort      Wii   
6                                 New Super Mario Bros.       DS   
7                                              Wii Play      Wii   
...                                                 ...      ...   
16947                     Men in Black II: Alien Escape       GC   
16949                Woody Woodpecker in Crazy Castle 5      GBA   
16950  SCORE International Baja 1000: The Official Game      PS2   
16952                                  LMA Manager 2007     X360   
16954                                  Spirits & Spells      GBA   

       year_of_release     genre  na_sales   eu_sales  jp_sales  other_sales  \
0                 2006    sports     41.36  28.959999      3.77         8.45   
2                 2008 

In [31]:
# разделяем все игры по оценкам критиков и выделяем категории:

def categorize_critic_score(grade):
    # низкая оценка (от 0 до 30, не включая правую границу интервала)
    if grade < 30:
        return 'Низкая оценка'  
    # средняя оценка (от 30 до 80, не включая правую границу интервала)
    elif grade < 80:
        return 'Средняя оценка'
    # высокая оценка (от 80 до 100 включительно)
    elif grade <= 100:
        return 'Высокая оценка'
    else:
        return 'Не определено'

df_actual['critic_score_category'] = df_actual['critic_score'].apply(categorize_critic_score)
print(df_actual)

                                                   name platform  \
0                                            Wii Sports      Wii   
2                                        Mario Kart Wii      Wii   
3                                     Wii Sports Resort      Wii   
6                                 New Super Mario Bros.       DS   
7                                              Wii Play      Wii   
...                                                 ...      ...   
16947                     Men in Black II: Alien Escape       GC   
16949                Woody Woodpecker in Crazy Castle 5      GBA   
16950  SCORE International Baja 1000: The Official Game      PS2   
16952                                  LMA Manager 2007     X360   
16954                                  Spirits & Spells      GBA   

       year_of_release     genre  na_sales   eu_sales  jp_sales  other_sales  \
0                 2006    sports     41.36  28.959999      3.77         8.45   
2                 2008 

In [32]:
# Проверка результата: группировка и подсчёт количества игр в каждой категории
print("=" * 60)
print("РЕЗУЛЬТАТЫ КАТЕГОРИЗАЦИИ ОЦЕНОК ПОЛЬЗОВАТЕЛЕЙ")
print("=" * 60)

user_category_counts = df_actual['user_score_category'].value_counts()
print("Количество игр по категориям оценок пользователей:")
print(user_category_counts)

user_percentages = df_actual['user_score_category'].value_counts(normalize=True) * 100
print("\nПроцентное соотношение игр по категориям (оценки пользователей):")
print(user_percentages.round(2))

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ КАТЕГОРИЗАЦИИ ОЦЕНОК КРИТИКОВ")
print("=" * 60)

critic_category_counts = df_actual['critic_score_category'].value_counts()
print("Количество игр по категориям оценок критиков:")
print(critic_category_counts)

critic_percentages = df_actual['critic_score_category'].value_counts(normalize=True) * 100
print("\nПроцентное соотношение игр по категориям (оценки критиков):")
print(critic_percentages.round(2))

РЕЗУЛЬТАТЫ КАТЕГОРИЗАЦИИ ОЦЕНОК ПОЛЬЗОВАТЕЛЕЙ
Количество игр по категориям оценок пользователей:
Низкая оценка     6414
Средняя оценка    4081
Высокая оценка    2286
Name: user_score_category, dtype: int64

Процентное соотношение игр по категориям (оценки пользователей):
Низкая оценка     50.18
Средняя оценка    31.93
Высокая оценка    17.89
Name: user_score_category, dtype: float64

РЕЗУЛЬТАТЫ КАТЕГОРИЗАЦИИ ОЦЕНОК КРИТИКОВ
Количество игр по категориям оценок критиков:
Низкая оценка     5667
Средняя оценка    5422
Высокая оценка    1692
Name: critic_score_category, dtype: int64

Процентное соотношение игр по категориям (оценки критиков):
Низкая оценка     44.34
Средняя оценка    42.42
Высокая оценка    13.24
Name: critic_score_category, dtype: float64


* Выделим топ-7 платформ по количеству игр, выпущенных за весь актуальный период.*

In [33]:
# Группируем данные по платформам и считаем количество игр для каждой
platform_counts = df_actual['platform'].value_counts()

# Берём топ-7 платформ
top_7_platforms = platform_counts.head(7)

print("ТОП-7 платформ по количеству игр (2000–2013 гг.):")
print("=" * 50)
print(top_7_platforms)

# Дополнительно: выводим в более читаемом формате с нумерацией
print("\nТОП-7 платформ (подробно):")
print("-" * 40)
for i, (platform, count) in enumerate(top_7_platforms.items(), 1):
    print(f"{i}. {platform}: {count} игр")

ТОП-7 платформ по количеству игр (2000–2013 гг.):
PS2     2127
DS      2120
Wii     1275
PSP     1180
X360    1121
PS3     1087
GBA      811
Name: platform, dtype: int64

ТОП-7 платформ (подробно):
----------------------------------------
1. PS2: 2127 игр
2. DS: 2120 игр
3. Wii: 1275 игр
4. PSP: 1180 игр
5. X360: 1121 игр
6. PS3: 1087 игр
7. GBA: 811 игр


---

## 5. Итоговый вывод

In [34]:
print(df_actual)

                                                   name platform  \
0                                            Wii Sports      Wii   
2                                        Mario Kart Wii      Wii   
3                                     Wii Sports Resort      Wii   
6                                 New Super Mario Bros.       DS   
7                                              Wii Play      Wii   
...                                                 ...      ...   
16947                     Men in Black II: Alien Escape       GC   
16949                Woody Woodpecker in Crazy Castle 5      GBA   
16950  SCORE International Baja 1000: The Official Game      PS2   
16952                                  LMA Manager 2007     X360   
16954                                  Spirits & Spells      GBA   

       year_of_release     genre  na_sales   eu_sales  jp_sales  other_sales  \
0                 2006    sports     41.36  28.959999      3.77         8.45   
2                 2008 

### Проделанная работа
В ходе анализа был выполнен комплекс операций с датасетом об играх. Разберём этапы подробно:

**Этап 1. Подготовка данных**

* Срез данных: из исходного датасета df_no_duplicates сформирован срез df_actual, включающий только записи за 2000–2013 гг.:

`df_actual = df_no_duplicates[df_no_duplicates['year_of_release'].between(2000, 2013)]`
Цель среза: сосредоточить анализ на актуальном 14‑летнем периоде, исключив исторические данные.

* Проверка качества: перед категоризацией проведена проверка на пропущенные значения (NaN) в столбцах user_score и critic_score.

**Этап 2. Категоризация оценок**

* Созданы две функции для преобразования числовых оценок в текстовые категории:

* `categorize_user_score(grade)` — для оценок пользователей (шкала 0–10):

 * «Низкая оценка»: grade<3;

 * «Средняя оценка»: 3≤grade<8;

 * «Высокая оценка»: 8≤grade≤10;

 * «Не определено»: остальные случаи (включая NaN).

* `categorize_critic_score(grade)` — для оценок критиков (шкала 0–100):

 * «Низкая оценка»: grade<30;

 * «Средняя оценка»: 30≤grade<80;

 * «Высокая оценка»: 80≤grade≤100;

 * «Не определено»: остальные случаи.

Функции применены к соответствующим столбцам через метод .apply(), что позволило категоризировать все записи.

**Новые поля в датасете:**

* games_categories_user_score — категории оценок пользователей;

* games_categories_critic_score — категории оценок критиков.

**Этап 3. Анализ результатов категоризации**

Для каждой категории выполнен подсчёт количества игр и расчёт процентного соотношения:

* использован метод `.value_counts()` для группировки и подсчёта;

* параметр normalize=True позволил получить доли, которые затем переведены в проценты.

**Этап 4. Выделение топ‑7 платформ**

Проведён анализ популярности игровых платформ:

* данные сгруппированы по столбцу platform;

* подсчитано количество игр для каждой платформы;

* выбраны 7 самых популярных платформ с помощью .head(7);

* дополнительно рассчитаны:

* абсолютное количество игр на каждой платформе;

* процент от общего числа игр, приходящийся на топ‑7.

**Этап 5. Визуализация и детализация**

* Представлены результаты в нескольких форматах:

* табличный вывод (краткий и подробный с нумерацией);

* столбчатая диаграмма для наглядного сравнения платформ;

* процентное соотношение топ‑7 к общему числу игр.

**Основной вывод**

Анализ данных об играх за 2000–2013 гг. позволил получить следующие ключевые результаты:

**Распределение оценок:**

* оценки пользователей и критиков имеют различную структуру распределения по категориям;

* выявлены игры с неопределёнными оценками (из‑за пропущенных данных)
**Популярность платформ:**

* определены 7 лидирующих платформ по количеству выпущенных игр;

* топ‑7 охватывает значительную долю рынка (точный процент зависит от данных);

* это позволяет сфокусироваться на ключевых платформах при планировании выпуска новых игр или маркетинге.

**Практическая ценность:**

* категоризация оценок упрощает интерпретацию числовых данных для нетехнических пользователей;

* рейтинг платформ помогает выявить целевые аудитории и приоритетные платформы для разработки;

* полученные данные могут быть использованы для:

* прогнозирования спроса на игры разных жанров;

* анализа корреляции между оценками и продажами;

* планирования релизов на конкретных платформах.

Таким образом, проведённый анализ дал структурированное представление о рынке видеоигр в 2000–2013 гг., выделив ключевые платформы и охарактеризовав распределение оценок пользователей и критиков. Эти выводы могут служить основой для стратегических решений в игровой индустрии.